# FFT Noise Analysis

In [ ]:
2 ** 13

In [ ]:
from scipy.fft import rfft, rfftfreq
from scipy.signal import get_window
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
SAMPLING_FREQUENCY = 2 ** 13
print(SAMPLING_FREQUENCY)
DURATION = 5
t = np.linspace(0, DURATION, SAMPLING_FREQUENCY * DURATION)
print(t.shape)
freq = 1000

signal = np.sin(2 * np.pi * freq * t)

noise_power = 0.01
noise = np.sqrt(noise_power) * np.random.normal(size=len(t))

mixed_signal = signal + noise

In [ ]:
n = len(mixed_signal)
# f = np.linspace(0, fs/2, n)
fft_values = rfft(mixed_signal)
magnitude = np.abs(fft_values)
mag_db = 20 * np.log10(magnitude)
f = rfftfreq(n, 1/SAMPLING_FREQUENCY)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, magnitude, "-")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude (V)")
# ax.set_xlim(0, 100)
# ax.set_ylim(0, 200)
# ax.set_xscale("log")
plt.grid()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, mag_db, "-")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude (dB)")
# ax.set_xlim(0, 100)
# ax.set_ylim(0, 200)
# ax.set_xscale("log")
plt.grid()

In [ ]:
n = len(noise)
# f = np.linspace(0, fs/2, n)
fft_values = rfft(noise)
magnitude = np.abs(fft_values)
mag_db = 20 * np.log10(magnitude)
f = rfftfreq(n, 1/SAMPLING_FREQUENCY)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, magnitude, ".")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude (V)")
# ax.set_xlim(0, 100)
# ax.set_ylim(0, 200)
# ax.set_xscale("log")
plt.grid()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, mag_db, ".")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude (dB)")
# ax.set_xlim(0, 100)
# ax.set_ylim(0, 200)
# ax.set_xscale("log")
plt.grid()

In [ ]:
# split into frames
RESOLUTION_LINES = 2 ** 12
frame_samples = 2 * RESOLUTION_LINES
print(frame_samples / SAMPLING_FREQUENCY)
n_frames = int(len(noise) / frame_samples)
print(len(noise))
print(n_frames)

frames = [noise[i*frame_samples:(i+1)*frame_samples] for i in range(n_frames)]
print(frames)

In [ ]:
# get fft per frame (using rfft), applying window
blackman_window = get_window("blackman", frame_samples)
# print(blackman_window)
windowed_frames = [frame*blackman_window for frame in frames]
frame_ffts = [rfft(frame) for frame in windowed_frames]
frame_ffts = [2 / frame_samples * np.abs(frame) for frame in frame_ffts]

In [ ]:
# square fft (multiply by complex conjugate, but since it's real, that's just squaring it)
squared_frames = [np.pow(fft, 2) for fft in frame_ffts]

In [ ]:
# find mean of all frames
all_ffts = np.stack(squared_frames, axis=1)
psd = np.mean(all_ffts, axis=1)
print(psd.shape)
print(psd)

In [ ]:
# normalize by dividing mean by sample rate
psd = psd / SAMPLING_FREQUENCY
psd_db = 10 * np.log10(psd)

In [ ]:
f = rfftfreq(frame_samples, 1/SAMPLING_FREQUENCY)
print(f.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, psd, "-")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (V)")
ax.set_xlim(100, 10000)
# ax.set_ylim(0, 200)
ax.set_xscale("log")
ax.set_yscale("log")
plt.grid()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(f, psd_db, "-")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (dB)")
ax.set_xlim(100, 10000)
# ax.set_ylim(0, 200)
ax.set_xscale("log")
# ax.set_yscale("log")
plt.grid()